# Open Domain QA Ensemble Voting

이 노트북은 여러 모델의 예측 결과를 soft voting과 hard voting 방식으로 앙상블합니다.

In [1]:
import pandas as pd
from collections import Counter
import numpy as np

## 1. CSV 파일 로드

In [ ]:
# CSV 파일 읽기 (탭으로 구분된 파일)
df1 = pd.read_csv('output (3).csv', sep='\t', header=None, names=['id', 'answer'])
df2 = pd.read_csv('output_hard_voting.csv', sep='\t', header=None, names=['id', 'answer'])
df3 = pd.read_csv('output_soft_voting.csv', sep='\t', header=None, names=['id', 'answer'])

print(f"Model 1 predictions: {len(df1)}")
print(f"Model 2 predictions: {len(df2)}")
print(f"Model 3 predictions: {len(df3)}")

# 샘플 데이터 확인
print("\n=== Sample from Model 1 ===")
print(df1.head())
print("\n=== Sample from Model 2 ===")
print(df2.head())
print("\n=== Sample from Model 3 ===")
print(df3.head())

Model 1 predictions: 600
Model 2 predictions: 600
Model 3 predictions: 600

=== Sample from Model 1 ===
             id         answer
0  mrc-1-000653       사만(サマーン)
1  mrc-1-001113             냉전
2  mrc-0-002191  대통령인 빌헬름 미클라스
3  mrc-0-003951           뉴질랜드
4  mrc-1-001272           프랑스군

=== Sample from Model 2 ===
             id    answer
0  mrc-1-000653     히스 행성
1  mrc-1-001113     냉전 종식
2  mrc-0-002191  빌헬름 미클라스
3  mrc-0-003951      뉴질랜드
4  mrc-1-001272       프랑스

=== Sample from Model 3 ===
             id         answer
0  mrc-1-000653          사만 행성
1  mrc-1-001113             냉전
2  mrc-0-002191  대통령인 빌헬름 미클라스
3  mrc-0-003951           뉴질랜드
4  mrc-1-001272            프랑스


## 2. Hard Voting

각 질문에 대해 가장 많이 예측된 답변을 선택합니다.

In [3]:
def hard_voting(df1, df2, df3):
    """
    Hard voting: 각 질문에 대해 가장 많이 나온 답변을 선택
    동점일 경우 첫 번째 모델의 답변 선택
    """
    results = []
    
    # ID가 동일한지 확인
    assert all(df1['id'] == df2['id']) and all(df1['id'] == df3['id']), "IDs must match across all dataframes"
    
    for idx in range(len(df1)):
        qid = df1.iloc[idx]['id']
        answers = [
            df1.iloc[idx]['answer'],
            df2.iloc[idx]['answer'],
            df3.iloc[idx]['answer']
        ]
        
        # 가장 많이 나온 답변 선택
        counter = Counter(answers)
        most_common = counter.most_common(1)[0][0]
        
        results.append({'id': qid, 'answer': most_common})
    
    return pd.DataFrame(results)

hard_voted_df = hard_voting(df1, df2, df3)
print("=== Hard Voting Results (샘플) ===")
print(hard_voted_df.head(10))
print(f"\nTotal predictions: {len(hard_voted_df)}")

=== Hard Voting Results (샘플) ===
             id         answer
0  mrc-1-000653       사만(サマーン)
1  mrc-1-001113             냉전
2  mrc-0-002191  대통령인 빌헬름 미클라스
3  mrc-0-003951           뉴질랜드
4  mrc-1-001272            프랑스
5  mrc-1-000993             유두
6  mrc-0-005021           민주주의
7  mrc-1-000163   하코네 산의 화산 폭발
8  mrc-0-001283   순조 11년(1811)
9  mrc-0-004543         고전도성 철

Total predictions: 600


## 3. Soft Voting (문자열 유사도 기반)

Open domain QA에서는 confidence score가 없으므로, 문자열 유사도를 기반으로 soft voting을 수행합니다.

In [4]:
from difflib import SequenceMatcher

def string_similarity(a, b):
    """두 문자열 간의 유사도를 계산 (0~1)"""
    return SequenceMatcher(None, a, b).ratio()

def soft_voting(df1, df2, df3, similarity_threshold=0.8):
    """
    Soft voting: 문자열 유사도를 기반으로 가중치를 부여
    유사한 답변들을 그룹화하여 가장 높은 점수를 받은 답변 선택
    """
    results = []
    
    assert all(df1['id'] == df2['id']) and all(df1['id'] == df3['id']), "IDs must match across all dataframes"
    
    for idx in range(len(df1)):
        qid = df1.iloc[idx]['id']
        answers = [
            df1.iloc[idx]['answer'],
            df2.iloc[idx]['answer'],
            df3.iloc[idx]['answer']
        ]
        
        # 각 답변에 대한 점수 계산 (자기 자신 포함)
        scores = {}
        for ans in answers:
            if ans not in scores:
                score = 0
                for other_ans in answers:
                    score += string_similarity(ans, other_ans)
                scores[ans] = score
        
        # 가장 높은 점수를 받은 답변 선택
        best_answer = max(scores.items(), key=lambda x: x[1])[0]
        
        results.append({'id': qid, 'answer': best_answer})
    
    return pd.DataFrame(results)

soft_voted_df = soft_voting(df1, df2, df3)
print("=== Soft Voting Results (샘플) ===")
print(soft_voted_df.head(10))
print(f"\nTotal predictions: {len(soft_voted_df)}")

=== Soft Voting Results (샘플) ===
             id          answer
0  mrc-1-000653           사만 행성
1  mrc-1-001113              냉전
2  mrc-0-002191   대통령인 빌헬름 미클라스
3  mrc-0-003951            뉴질랜드
4  mrc-1-001272             프랑스
5  mrc-1-000993              유두
6  mrc-0-005021            민주주의
7  mrc-1-000163  시공을 초월한 영원한 존재
8  mrc-0-001283    순조 11년(1811)
9  mrc-0-004543          고전도성 철

Total predictions: 600


## 4. 결과 비교

In [5]:
# Hard voting과 Soft voting 결과 비교
comparison = pd.DataFrame({
    'id': df1['id'],
    'model_1': df1['answer'],
    'model_2': df2['answer'],
    'model_3': df3['answer'],
    'hard_voting': hard_voted_df['answer'],
    'soft_voting': soft_voted_df['answer']
})

print("=== 앙상블 결과 비교 (처음 20개) ===")
print(comparison.head(20))

# Hard voting과 Soft voting이 다른 경우 찾기
different = comparison[comparison['hard_voting'] != comparison['soft_voting']]
print(f"\nHard voting과 Soft voting이 다른 경우: {len(different)}개")
if len(different) > 0:
    print("\n=== 다른 결과 샘플 ===")
    print(different.head(10))

=== 앙상블 결과 비교 (처음 20개) ===
              id           model_1           model_2            model_3  \
0   mrc-1-000653          사만(サマーン)             히스 행성              사만 행성   
1   mrc-1-001113                냉전             냉전 종식                 냉전   
2   mrc-0-002191     대통령인 빌헬름 미클라스          빌헬름 미클라스      대통령인 빌헬름 미클라스   
3   mrc-0-003951              뉴질랜드              뉴질랜드               뉴질랜드   
4   mrc-1-001272              프랑스군               프랑스                프랑스   
5   mrc-1-000993                유두           유대류의 새끼                아래턱   
6   mrc-0-005021              민주주의              대의제도               민주주의   
7   mrc-1-000163      하코네 산의 화산 폭발    시공을 초월한 영원한 존재  광독 가스 및 그에 따른 산성비   
8   mrc-0-001283      순조 11년(1811)      순조 11년(1811)       순조 11년(1811)   
9   mrc-0-004543            고전도성 철            고전도성 철             고전도성 철   
10  mrc-0-000439            리처드 말킨            리처드 말킨                점쟁이   
11  mrc-0-002895  칼라치 전방 약250km 지점  칼라치 전방 약250km 지점   칼라치 전방 약250km 지점  

## 5. 최종 결과 저장

In [ ]:
# Hard voting 결과 저장
hard_voted_df.to_csv('output_hard_voting_v2.csv', sep='\t', index=False, header=False)
print("Hard voting 결과가 'output_hard_voting.csv'에 저장되었습니다.")

# Soft voting 결과 저장
soft_voted_df.to_csv('output_soft_voting_v2.csv', sep='\t', index=False, header=False)
print("Soft voting 결과가 'output_soft_voting.csv'에 저장되었습니다.")

print(f"\n총 {len(hard_voted_df)}개의 예측이 생성되었습니다.")

Hard voting 결과가 'output_hard_voting.csv'에 저장되었습니다.
Soft voting 결과가 'output_soft_voting.csv'에 저장되었습니다.

총 600개의 예측이 생성되었습니다.


## 6. 통계 분석

In [7]:
# 모델 간 일치도 분석
print("=== 모델 간 일치도 분석 ===")

# 3개 모델이 모두 일치하는 경우
all_agree = sum((df1['answer'] == df2['answer']) & (df2['answer'] == df3['answer']))
print(f"3개 모델 모두 일치: {all_agree}개 ({all_agree/len(df1)*100:.2f}%)")

# 2개 모델이 일치하는 경우
two_agree = sum(
    ((df1['answer'] == df2['answer']) & (df2['answer'] != df3['answer'])) |
    ((df1['answer'] == df3['answer']) & (df1['answer'] != df2['answer'])) |
    ((df2['answer'] == df3['answer']) & (df2['answer'] != df1['answer']))
)
print(f"2개 모델 일치: {two_agree}개 ({two_agree/len(df1)*100:.2f}%)")

# 모두 다른 경우
all_different = sum((df1['answer'] != df2['answer']) & (df2['answer'] != df3['answer']) & (df1['answer'] != df3['answer']))
print(f"모두 다름: {all_different}개 ({all_different/len(df1)*100:.2f}%)")

# 각 모델별 답변 길이 통계
print("\n=== 답변 길이 통계 ===")
print(f"Model 1 평균 답변 길이: {df1['answer'].str.len().mean():.2f}자")
print(f"Model 2 평균 답변 길이: {df2['answer'].str.len().mean():.2f}자")
print(f"Model 3 평균 답변 길이: {df3['answer'].str.len().mean():.2f}자")
print(f"Hard voting 평균 답변 길이: {hard_voted_df['answer'].str.len().mean():.2f}자")
print(f"Soft voting 평균 답변 길이: {soft_voted_df['answer'].str.len().mean():.2f}자")

=== 모델 간 일치도 분석 ===
3개 모델 모두 일치: 306개 (51.00%)
2개 모델 일치: 188개 (31.33%)
모두 다름: 106개 (17.67%)

=== 답변 길이 통계 ===
Model 1 평균 답변 길이: 6.22자
Model 2 평균 답변 길이: 6.35자
Model 3 평균 답변 길이: 7.08자
Hard voting 평균 답변 길이: 6.20자
Soft voting 평균 답변 길이: 6.39자
